In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 1. SETUP ENVIRONMENT CONFIGURATION
start = np.array([0.0, 0.0])
goal = np.array([10.0, 10.0])
robot_pos = np.copy(start)

# Obstacle starting positions and their constant velocities [x, y, vx, vy]
obstacles = [
    {"pos": np.array([3.0, 2.0]), "vel": np.array([0.1, 0.3]), "radius": 1.2},
    {"pos": np.array([7.0, 8.0]), "vel": np.array([-0.2, -0.1]), "radius": 1.0},
    {"pos": np.array([5.0, 5.0]), "vel": np.array([0.0, -0.2]), "radius": 1.5}
]

# 2. POTENTIAL FIELD PARAMETERS
K_ATTRACT = 0.5   # Attraction strength to goal
K_REPEL = 15.0    # Repulsion strength from obstacles
INFLUENCE_DIST = 2.5 # How close obstacle must be to repel
STEP_SIZE = 0.15  # Distance robot moves per iteration
MAX_STEPS = 150

# Track data for animation
robot_history = [np.copy(robot_pos)]
obstacle_histories = [[np.copy(obs["pos"])] for obs in obstacles]

# 3. RUN SIMULATION LOOP
for step in range(MAX_STEPS):
    # Check if goal reached
    if np.linalg.norm(robot_pos - goal) < 0.3:
        break

    # Move obstacles for this step
    for i, obs in enumerate(obstacles):
        obs["pos"] += obs["vel"]
        # Keep obstacles roughly within bounds
        if obs["pos"][0] < 0 or obs["pos"][0] > 11: obs["vel"][0] *= -1
        if obs["pos"][1] < 0 or obs["pos"][1] > 11: obs["vel"][1] *= -1
        obstacle_histories[i].append(np.copy(obs["pos"]))

    # Calculate Attractive Force Vector (Pulling toward goal)
    f_attract = K_ATTRACT * (goal - robot_pos)

    # Calculate Repulsive Force Vector (Pushing away from obstacles)
    f_repel = np.array([0.0, 0.0])
    for obs in obstacles:
        vec_to_robot = robot_pos - obs["pos"]
        dist = np.linalg.norm(vec_to_robot)

        if dist <= INFLUENCE_DIST:
            # Stronger push the closer the robot gets
            if dist < 0.1: dist = 0.1 # Prevent division by zero
            f_repel += K_REPEL * (1.0/dist - 1.0/INFLUENCE_DIST) * (1.0/(dist**2)) * (vec_to_robot / dist)

    # Total Force determines movement direction
    total_force = f_attract + f_repel
    force_magnitude = np.linalg.norm(total_force)

    if force_magnitude > 0:
        step_dir = total_force / force_magnitude
        robot_pos += step_dir * STEP_SIZE

    robot_history.append(np.copy(robot_pos))

# 4. GENERATE ANIMATION IN JUPYTER/COLAB
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 11)
ax.grid(True)
ax.set_title("Reactive Obstacle Avoidance (Artificial Potential Field)")

# Plot static points
ax.plot(start[0], start[1], 'go', markersize=10, label='Start')
ax.plot(goal[0], goal[1], 'ro', markersize=10, label='Goal')

# Create dynamic visual elements
robot_dot, = ax.plot([], [], 'bo', markersize=8, label='Robot')
robot_path, = ax.plot([], [], 'b--', alpha=0.6)
obs_circles = [plt.Circle((0,0), obs["radius"], color='black', alpha=0.4) for obs in obstacles]
for circle in obs_circles:
    ax.add_patch(circle)

ax.legend(loc='upper left')

def update(frame):
    # Update robot position
    r_pos = robot_history[min(frame, len(robot_history)-1)]
    robot_dot.set_data([r_pos[0]], [r_pos[1]])

    # Update robot historical trail
    trail = np.array(robot_history[:frame+1])
    if len(trail) > 0:
        robot_path.set_data(trail[:, 0], trail[:, 1])

    # Update obstacle positions
    for i, circle in enumerate(obs_circles):
        obs_pos_hist = obstacle_histories[i]
        o_pos = obs_pos_hist[min(frame, len(obs_pos_hist)-1)]
        circle.center = (o_pos[0], o_pos[1])

    return [robot_dot, robot_path] + obs_circles

ani = FuncAnimation(fig, update, frames=max(len(robot_history), len(obstacle_histories[0])), interval=50, blit=True)
plt.close()

# Render video block in browser
HTML(ani.to_jshtml())